In [1]:
import json 
import pandas as pd 
from glob import glob 

model_names = [
            "OpenGVLab/InternVL3_5-8B",
            "OpenGVLab/InternVL3_5-2B",
            "OpenGVLab/InternVL3_5-4B",
            "OpenGVLab/InternVL3_5-1B",

            "Qwen/Qwen3-VL-2B-Instruct", 
            "Qwen/Qwen3-VL-4B-Instruct", 
            "Qwen/Qwen3-VL-8B-Instruct",

            "llava-hf/llava-1.5-7b-hf", 
            "llava-hf/llava-v1.6-vicuna-7b-hf", 
            "llava-hf/llava-v1.6-mistral-7b-hf", 

            "Qwen/Qwen3-8B-Base", 
            "Qwen/Qwen3-4B-Base", 
            "Qwen/Qwen3-1.7B-Base" , 
            "Qwen/Qwen3-0.6B-Base" # doesnt work 
]

files = glob("/home/work/yuna/HPA/results/swift-results/*.jsonl")  # glob("/home/work/yuna/HPA/results/swift-results/*/*.jsonl")+ 
len(files)

88

In [2]:
import os 
import shutil 
conditions =['sys_inst_blind', 'inst_blind', 'blind', ''] 
indicators = ['pid', 'id', 'image_id', 'index']  
check_dir='/home/work/yuna/HPA/results/check' 

dfs = []
for f in files:
    if 'vqav2_val_1k' in f or 'syst' in f: 
        dest_path = os.path.join(check_dir, f.split('/')[-1])
        if os.path.isfile(f):
            shutil.move(f, dest_path)
            print(f"Moved: {f}")
            continue 
    
    try: 
        df = pd.read_json(f, lines=True)
    except Exception as e : 
        print(e, f)
        continue

    f = f.replace("_vlm", '').replace("_llm", '')  
    df['filename'] = f
    
    hashable_cols = []
    for col in df.columns:
        try:
            # Try to hash the first non-null value
            sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
            if sample is not None:
                hash(sample)
            hashable_cols.append(col)
        except (TypeError, KeyError):
            pass

    # Drop duplicates based on hashable columns
    if hashable_cols:
        df = df.drop_duplicates(subset=hashable_cols)
    else:
        # Fallback: convert entire rows to JSON strings
        df['_temp_hash'] = df.apply(lambda row: json.dumps(row.to_dict(), sort_keys=True, default=str), axis=1)
        df = df.drop_duplicates(subset=['_temp_hash']).drop(columns=['_temp_hash'])
        
    # df.to_json(f, orient='records', lines=True)
    # model = f.split('/')[-1][:-6]
    model_name = None 
    for model in model_names : 
        model_name = model.split('/')[-1] 
        if model_name in f :   
            df['model'] = model_name  
            f = f.replace(f'{model_name}_', '') 
            break 
    if model_name is None : 
        print(f, 'model is not found')

    condition= '' 
    for c in conditions : 
        if c in f : 
            f = f.replace(f'_{c}', '')  
            condition = c 
            break 
    df['dataset'] = f.split('/')[-1][:-6] # .split('_')[1].split('_')[0]
    df['condition'] = condition

    if "response" not in df.columns: 
        df.rename(columns={'output':'response'}, inplace=True)
    # else: 
    dfs.append(df) 

In [3]:

df = pd.concat(dfs)
summary = df.groupby(['dataset', 'condition', 'model', 'filename']).count()['response'].reset_index().sort_values(by=['response', 'model'])


In [4]:
summary = summary.pivot_table(index=['model'], columns=['dataset', 'condition' , ], values=['response'])
summary.to_csv('./inference_progress.csv') # , 'dataset', 'condition'
summary

response                                         \
dataset                    mmstar                    spubench              
condition                           blind inst_blind          inst_blind   
model                                                                      
InternVL3_5-1B             1500.0     NaN        NaN   2400.0     2400.0   
InternVL3_5-2B             1500.0  1500.0        NaN   2400.0     2400.0   
InternVL3_5-4B             1500.0  1500.0     1500.0   2400.0     2400.0   
InternVL3_5-8B             1500.0     NaN     1500.0   2400.0     2400.0   
Qwen3-0.6B-Base            1500.0     NaN        NaN   2400.0        NaN   
Qwen3-1.7B-Base            1500.0     NaN        NaN   2400.0        NaN   
Qwen3-4B-Base              1500.0     NaN        NaN   2400.0        NaN   
Qwen3-8B-Base              1500.0     NaN        NaN      NaN        NaN   
Qwen3-VL-2B-Instruct          NaN  1500.0        NaN   2400.0     2400.0   
Qwen3-VL-4B-Instruct       1500.0  1500.0     1500.0   2400.0     2400.0   
Qwen3-VL-8B-Instruct        534.0     NaN     1500.0   2400.0     2400.0   
llava-1.5-7b-hf             775.0     NaN      775.0   2400.0     2400.0   
llava-v1.6-mistral-7b-hf   1500.0     NaN     1500.0      NaN     2400.0   
llava-v1.6-vicuna-7b-hf    1500.0     NaN     1500.0      NaN     2400.0   

                                                                            \
dataset                    vqa1k   vqa5k  vqa_1k                             
condition                                  blind inst_blind sys_inst_blind   
model                                                                        
InternVL3_5-1B            1000.0  5000.0  1000.0     1000.0            NaN   
InternVL3_5-2B               NaN  5000.0  1000.0        NaN            NaN   
InternVL3_5-4B            1000.0  5454.0  1000.0     1000.0         1000.0   
InternVL3_5-8B            1000.0  5000.0     NaN     1000.0            NaN   
Qwen3-0.6B-Base           1000.0     NaN     NaN        NaN            NaN   
Qwen3-1.7B-Base           1000.0     NaN     NaN        NaN            NaN   
Qwen3-4B-Base                NaN     NaN     NaN        NaN            NaN   
Qwen3-8B-Base             1000.0     NaN     NaN        NaN            NaN   
Qwen3-VL-2B-Instruct         NaN  5000.0     NaN        NaN            NaN   
Qwen3-VL-4B-Instruct      1000.0  5000.0  1000.0     1000.0         1000.0   
Qwen3-VL-8B-Instruct      1000.0  6424.0     NaN     1000.0            NaN   
llava-1.5-7b-hf           1000.0  5000.0     NaN     1000.0            NaN   
llava-v1.6-mistral-7b-hf     NaN  5000.0     NaN     1000.0            NaN   
llava-v1.6-vicuna-7b-hf      NaN  5000.0     NaN     1000.0            NaN   

                                             
dataset                   vqa_5k             
condition                  blind inst_blind  
model                                        
InternVL3_5-1B            5000.0        NaN  
InternVL3_5-2B            5000.0        NaN  
InternVL3_5-4B            5000.0        NaN  
InternVL3_5-8B               NaN        NaN  
Qwen3-0.6B-Base              NaN        NaN  
Qwen3-1.7B-Base              NaN        NaN  
Qwen3-4B-Base                NaN        NaN  
Qwen3-8B-Base                NaN        NaN  
Qwen3-VL-2B-Instruct      5000.0        NaN  
Qwen3-VL-4B-Instruct      5000.0        NaN  
Qwen3-VL-8B-Instruct         NaN        NaN  
llava-1.5-7b-hf            912.0     5000.0  
llava-v1.6-mistral-7b-hf  5000.0        NaN  
llava-v1.6-vicuna-7b-hf   5000.0        NaN